# 🔥 05_burn_prob_rps_calc.ipynb
================================================================================

**Script 05 of 08** in the Networks Paper analysis pipeline.

## 🎯 Purpose
Calculate wildfire risk metrics (burn probability and RPS) for each CDP. These metrics quantify how likely each community is to experience wildfire.

## 🔥 Why This Matters for Wildfire Research
Understanding wildfire RISK combined with evacuation CAPACITY is the core of this research:
- **Burn Probability**: How likely is a wildfire to burn through this area?
- **RPS**: If a fire occurs, how much damage could it cause?

Communities with HIGH burn probability AND LIMITED egress routes are the most vulnerable.

## 📊 What are Burn Probability and RPS?

### Burn Probability (BP)
- **Definition**: Annual probability that a pixel will burn
- **Range**: 0.0 to 1.0 (0% to 100% chance per year)
- **Source**: USFS Wildfire Risk to Communities dataset
- **Resolution**: 270m (resampled to CONUS)

### Risk to Potential Structures (RPS)
- **Definition**: Expected annual loss if a structure existed at that location
- **Combines**: Burn probability × Fire intensity × Suppression difficulty
- **Units**: Expected flame length weighted by probability
- **Use**: Identifies where new development would face highest risk

## 📋 Overall Workflow
1. **Unzip** state-specific wildfire risk data files (Step 1)
2. **Burn Probability**: Compute zonal stats for all CDPs (Step 2)
3. **RPS**: Compute zonal stats for all CDPs (Step 3)
4. **Export** combined CSV files and state visualization maps

---

## 📥 INPUTS (Required Data Sources)
| Source | Path | Description |
|--------|------|-------------|
| Raw ZIP Files | `.../wildfire-risk-layers/raw-zip-files/` | State-specific risk data (download first) |
| CDP Shapefiles | `.../us-census-designated-places/{State}_{FIPS}/` | Place boundaries (from step 01) |
| CONUS Burn Prob | `.../BP_CONUS/BP_CONUS.tif` | Continental US burn probability |
| CONUS RPS | `.../RPS_CONUS/RPS_CONUS.tif` | Continental US risk to structures |

## 📤 OUTPUTS (Generated Files)
| Output | Path | Description |
|--------|------|-------------|
| Burn Prob CSV | `.../output_csvs/burn_prob_cdp/burn_prob_cdps_all_states.csv` | Mean, min, max, std burn prob per CDP |
| RPS CSV | `.../output_csvs/rps_cdp/rps_cdps_all_states.csv` | Mean, min, max, std RPS per CDP |
| State Maps | `.../output_maps/_state_wide_maps/` | Choropleth maps by state |

## ⏱️ Expected Runtime
- Unzip: ~10 minutes
- Burn Probability: ~30-60 minutes (60 cores)
- RPS: ~30-60 minutes (60 cores)

---

### 🗜️ Unzip Burn Probability files
================================

- **Purpose**: Extract each state’s wildfire risk ZIP into a per-state folder. Skips already-extracted items (idempotent).
- **Parallelism**: Uses a process pool with **50 workers** for faster I/O-bound extraction.

- **Inputs (paths)**:
  - `source_folder`: `{NETWORKS_RASTERS_DIR}/raw-zip-files`
- **Outputs (paths)**:
  - `dest_folder`: `{NETWORKS_RASTERS_DIR}/unzipped-states`

- **How it works**:
  1. List `*.zip` files in `source_folder`.
  2. For each file, create an output directory named after the ZIP (without `.zip`).
  3. If the output directory exists and is non-empty, skip (✅ already extracted).
  4. Otherwise, extract the ZIP contents into that directory.
  5. Run steps 2–4 concurrently with 50 workers.

- **Adjustments**:
  - Increase/decrease workers by editing `ProcessPoolExecutor(max_workers=50)` in the code below.
  - Paths can be changed at the top of the code block (`source_folder`, `dest_folder`).

In [ ]:
import os
import zipfile
from concurrent.futures import ProcessPoolExecutor, as_completed

# ── Configure paths ────────────────────────────────────────────────────────────
# Set NETWORKS_RASTERS_DIR to the folder containing wildfire risk ZIP files, or
# edit the default path below.
#   export NETWORKS_RASTERS_DIR="/your/path/to/wildfire-risk-layers"
RASTERS_DATA_DIR = os.environ.get("NETWORKS_RASTERS_DIR", os.path.expanduser("~/data/wildfire-risk-layers"))

# Define source and destination folder paths
source_folder = os.path.join(RASTERS_DATA_DIR, "raw-zip-files")
dest_folder = os.path.join(RASTERS_DATA_DIR, "unzipped-states")

def unzip_file(file_name):
    """
    Unzips the given zip file from the source folder if it hasn't been unzipped already,
    and saves the extracted content in the destination folder.
    """
    # Full path for the zip file in the source folder
    zip_path = os.path.join(source_folder, file_name)
    # Extraction folder (named after the zip file without the .zip extension) inside the destination folder
    extract_path = os.path.join(dest_folder, file_name[:-4])
    
    # Check if extraction folder exists and is not empty (i.e., already unzipped)
    if os.path.exists(extract_path) and os.listdir(extract_path):
        print(f"Skipping {file_name}: already extracted.")
        return
    
    # Ensure the extraction directory exists
    os.makedirs(extract_path, exist_ok=True)
    
    try:
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_path)
        print(f"Extracted: {file_name} -> {extract_path}")
    except zipfile.BadZipFile:
        print(f"Error: {file_name} is not a valid zip file.")

if __name__ == '__main__':
    # Verify that the source folder exists
    if not os.path.exists(source_folder):
        print(f"Error: The folder {source_folder} does not exist.")
        exit(1)
    
    # Create the destination folder if it doesn't exist
    os.makedirs(dest_folder, exist_ok=True)
    
    # List all files in the source folder and filter for .zip files
    all_files = os.listdir(source_folder)
    zip_files = [f for f in all_files if f.endswith(".zip")]
    
    # Use ProcessPoolExecutor for parallel processing with 10 workers
    with ProcessPoolExecutor(max_workers=50) as executor:
        futures = [executor.submit(unzip_file, file_name) for file_name in zip_files]
        # Wait for all futures to complete
        for future in as_completed(futures):
            future.result()


### 🔥 Burn Probability zonal statistics for CDPs
=============================================

- **Goal**: Compute burn probability statistics for every Census-designated place (CDP) and export results + maps.

- **Settings**:
  - `NUM_CORES`: number of parallel workers
  - Paths for rasters, shapefiles, CSV, and map outputs

- **Data sources**:
  - CONUS raster: `BP_CONUS.tif` for the contiguous U.S.
  - State-specific rasters for `Alaska`, `Hawaii`, `District_of_Columbia`

- **Workflow**:
  1. Load state boundaries (for plotting overlays).
  2. For each state folder (e.g., `Alabama_01`), locate `tl_2023_*_place.shp` (CDPs).
  3. Pick the correct raster: state-specific if needed; otherwise **CONUS**.
  4. Reproject CDP geometries to raster CRS if required.
  5. Run `zonal_stats` to compute: `mean, min, max, std, count, median` per CDP.
  6. Plot and save a state map colored by CDP `mean` burn probability.
  7. Save per-state stats and combine into a single CSV: `burn_prob_cdps_all_states.csv`.

- **Outputs**:
  - Maps in `output_map_dir` named `burn_prob_cdps_<State>.png`
  - Combined CSV in `output_csv_dir` named `burn_prob_cdps_all_states.csv`

- **Adjustments**:
  - Change cores via `NUM_CORES` in the settings block.
  - Update any path variables at the top of the code block as needed.

- **Notes**:
  - CRS is harmonized to the raster CRS to ensure valid statistics.
  - Missing state boundary overlays are logged as warnings (does not stop processing).

In [ ]:
################################################################################
#                            📦 IMPORT LIBRARIES                               #
################################################################################

import os
import glob
import re
import pandas as pd
import geopandas as gpd
import rasterio
import matplotlib.pyplot as plt
from rasterstats import zonal_stats
from multiprocessing import Pool
from datetime import datetime
import pytz

################################################################################
#                            🔧 SETTINGS & CONFIGURATION                       #
################################################################################

# =============================================================================
# ⏰ Timezone Configuration (Pacific Time for timestamps)
# =============================================================================
TIMEZONE = pytz.timezone('America/Los_Angeles')

def print_timestamped(message):
    """Print message with Pacific Time timestamp."""
    timestamp = datetime.now(TIMEZONE).strftime('%Y-%m-%d %H:%M:%S %Z')
    print(f"[{timestamp}] {message}")

# =============================================================================
# 🖥️ Parallel Processing Settings
# =============================================================================
NUM_CORES = 60  # Number of cores to use for parallel processing

################################################################################
#                            📂 FILE PATHS                                      #
################################################################################
# ┌─────────────────────────────────────────────────────────────────────────┐
# │  Configure paths to point to your data directories.                    │
# │  Override with environment variables before running:                    │
# │      export NETWORKS_DATA_DIR="/your/path/to/data"                      │
# │      export NETWORKS_RASTERS_DIR="/your/path/to/wildfire-risk-layers"  │
# │  Or edit the default paths below.                                       │
# └─────────────────────────────────────────────────────────────────────────┘
BASE_DATA_DIR   = os.environ.get("NETWORKS_DATA_DIR",    os.path.expanduser("~/data/networks_paper"))
RASTERS_DATA_DIR = os.environ.get("NETWORKS_RASTERS_DIR", os.path.expanduser("~/data/wildfire-risk-layers"))

# =============================================================================
# 📥 INPUT PATHS
# =============================================================================
raster_root    = os.path.join(RASTERS_DATA_DIR, "unzipped-states")
shapefile_root = os.path.join(BASE_DATA_DIR, "us-census-designated-places")

# CONUS and state-specific burn probability rasters
conus_raster_path  = os.path.join(raster_root, "RDS-2020-0016-2__BP_CONUS/BP_CONUS/BP_CONUS.tif")
alaska_raster_path = os.path.join(raster_root, "RDS-2020-0016-2_Alaska/AK/BP_AK.tif")
dc_raster_path     = os.path.join(raster_root, "RDS-2020-0016-2_DistrictOfColumbia/DC/BP_DC.tif")
hawaii_raster_path = os.path.join(raster_root, "RDS-2020-0016-2_Hawaii/RDS-2020-0016-2_Hawaii/HI/BP_HI.tif")

# Dictionary mapping non-CONUS states to their raster paths
state_raster_map = {
    "Alaska": alaska_raster_path,
    "District_of_Columbia": dc_raster_path,
    "Hawaii": hawaii_raster_path
}
# All other states will use the CONUS raster

# =============================================================================
# 📤 OUTPUT PATHS
# =============================================================================
output_csv_dir = os.path.join(BASE_DATA_DIR, "output_csvs", "burn_prob_cdp")
output_map_dir = os.path.join(BASE_DATA_DIR, "output_maps", "_state_wide_maps")

os.makedirs(output_csv_dir, exist_ok=True)
os.makedirs(output_map_dir, exist_ok=True)

# -------------------------------
# Load U.S. state boundaries from the Census TIGER/Line shapefile (500k resolution)
# -------------------------------
state_boundaries_url = "https://www2.census.gov/geo/tiger/GENZ2018/shp/cb_2018_us_state_500k.zip"
try:
    states_gdf = gpd.read_file(state_boundaries_url)
    print("Loaded state boundaries from Census TIGER/Line.")
except Exception as e:
    print("Error loading state boundaries:", e)
    states_gdf = None

# We'll share the states boundaries with worker processes via a global variable.
def init_worker(shared_states):
    global global_states_gdf
    global_states_gdf = shared_states

# -------------------------------
# Define the processing function for one state folder
# -------------------------------
def process_state(state_dir):
    try:
        # Expect folder names like "Alabama_01" or "District_of_Columbia_11"
        state_folder_name = os.path.basename(state_dir)
        m = re.match(r"^(.*)_[0-9]+$", state_folder_name)
        if not m:
            print(f"Skipping folder {state_dir}: name does not match expected pattern.")
            return None

        # Extract state name and create display and matching versions.
        state_name = m.group(1)                      # e.g., "Alabama" or "District_of_Columbia"
        state_display_name = state_name.replace("_", " ")
        
        # -------------------------------
        # Find the shapefile for Census designated places in this folder.
        # -------------------------------
        shapefile_pattern = os.path.join(state_dir, "tl_2023_*_place.shp")
        shapefile_list = glob.glob(shapefile_pattern)
        if not shapefile_list:
            print(f"No shapefile found in {state_dir}. Skipping.")
            return None
        shapefile_path = shapefile_list[0]
        
        # -------------------------------
        # Determine which raster file to use for this state.
        # -------------------------------
        if state_name in state_raster_map:
            raster_path = state_raster_map[state_name]
        else:
            # Use the CONUS raster for all other states
            raster_path = conus_raster_path
        
        if not os.path.exists(raster_path):
            print(f"Raster file {raster_path} not found for state {state_name}. Skipping.")
            return None
        
        print(f"\nProcessing state: {state_display_name}")
        print(f"  Shapefile: {shapefile_path}")
        print(f"  Raster: {raster_path}")
        
        # -------------------------------
        # Load the CDP shapefile and reproject if needed.
        # -------------------------------
        try:
            gdf = gpd.read_file(shapefile_path)
        except Exception as e:
            print(f"Error reading shapefile {shapefile_path}: {e}")
            return None

        try:
            with rasterio.open(raster_path) as src:
                raster_crs = src.crs
                nodata_value = src.nodata
        except Exception as e:
            print(f"Error opening raster {raster_path}: {e}")
            return None

        if gdf.crs != raster_crs:
            gdf = gdf.to_crs(raster_crs)
            print("  Reprojected CDP shapefile to match raster CRS.")
        
        # -------------------------------
        # Compute zonal statistics.
        # -------------------------------
        stats = zonal_stats(
            gdf,
            raster_path,
            stats="mean min max std count median",
            nodata=nodata_value,
            geojson_out=True
        )
        if not stats:
            print(f"  No zonal stats returned for state {state_display_name}. Skipping.")
            return None
        gdf_stats = gpd.GeoDataFrame.from_features(stats)
        
        # -------------------------------
        # Plot the results with state boundary overlay.
        # -------------------------------
        fig, ax = plt.subplots(1, 1, figsize=(10, 8))
        cmap = "viridis"
        gdf_stats.plot(column="mean", cmap=cmap, legend=True, ax=ax,
                       legend_kwds={'label': "Mean Burn Probability", 'shrink': 0.6})
        
        # Overlay state boundary using the global_states_gdf loaded by the initializer.
        if global_states_gdf is not None:
            state_boundary = global_states_gdf[global_states_gdf["NAME"].str.lower() == state_display_name.lower()]
            if not state_boundary.empty:
                if state_boundary.crs != raster_crs:
                    state_boundary = state_boundary.to_crs(raster_crs)
                state_boundary.boundary.plot(ax=ax, edgecolor="black", linewidth=2)
            else:
                print(f"  Warning: No state boundary found for {state_display_name}.")
        
        # Set a professional title with a newline to prevent overlap.
        ax.set_title(f"Mean Burn Probability for Census designated places\nin {state_display_name}",
                     fontsize=16, pad=15)
        ax.set_axis_off()  # Clean look
        
        # Save the map to file.
        state_name_no_underscore = state_name.replace("_", "")
        map_output_fp = os.path.join(output_map_dir, f"burn_prob_cdps_{state_name_no_underscore}.png")
        plt.savefig(map_output_fp, dpi=300, bbox_inches="tight")
        plt.close(fig)
        print(f"  Map saved to {map_output_fp}")
        
        # -------------------------------
        # Prepare zonal stats DataFrame for combining later.
        # -------------------------------
        cols_to_keep = ['GEOID', 'min', 'max', 'mean', 'std', 'count', 'median']
        missing_cols = [col for col in cols_to_keep if col not in gdf_stats.columns]
        if missing_cols:
            print(f"  Warning: missing columns {missing_cols} in zonal stats for {state_display_name}. Skipping state CSV export.")
            return None
        df_final = gdf_stats[cols_to_keep].copy()

        # Rename the zonal stats columns with prefix "bp_"
        df_final.rename(columns={
            'min': 'bp_min',
            'max': 'bp_max',
            'mean': 'bp_mean',
            'std': 'bp_std',
            'count': 'bp_count',
            'median': 'bp_median'
        }, inplace=True)

        df_final["state"] = state_display_name
        
        # Return results: the DataFrame, state name, and map file path.
        return (df_final, state_display_name, map_output_fp)
    
    except Exception as ex:
        print(f"Error processing {state_dir}: {ex}")
        return None

# -------------------------------
# Main processing block
# -------------------------------
if __name__ == '__main__':
    # Gather state directories from the shapefile root.
    state_dirs = [os.path.join(shapefile_root, d) for d in os.listdir(shapefile_root)
                  if os.path.isdir(os.path.join(shapefile_root, d))]
    
    # Create a multiprocessing Pool and initialize each worker with the state boundaries.
    with Pool(NUM_CORES, initializer=init_worker, initargs=(states_gdf,)) as pool:
        results = pool.map(process_state, state_dirs)
    
    # Filter out any results that returned None.
    valid_results = [res for res in results if res is not None]
    
    # Combine DataFrames from each state into one large CSV.
    if valid_results:
        df_list = [res[0] for res in valid_results]
        combined_df = pd.concat(df_list, ignore_index=True)
        combined_csv = os.path.join(output_csv_dir, "burn_prob_cdps_all_states.csv")
        combined_df.to_csv(combined_csv, index=False)
        print(f"\nCombined CSV saved to {combined_csv}")
    else:
        print("No state data processed.")


### 🏠 Risk to Potential Structures (RPS) zonal statistics for CDPs
===============================================================

- **Goal**: Compute RPS statistics for CDPs and export combined results + state maps.

- **Settings**:
  - `NUM_CORES`: number of parallel workers
  - Paths for rasters, shapefiles, CSV, and map outputs

- **Data sources**:
  - CONUS raster: `RPS_CONUS.tif`
  - State-specific rasters for `Alaska`, `Hawaii`, `District_of_Columbia`

- **Workflow**:
  1. Load state boundaries for a map overlay.
  2. For each state folder, find `tl_2023_*_place.shp` (CDPs).
  3. Choose the correct raster (state-specific vs. CONUS).
  4. Reproject CDP geometries to raster CRS if needed.
  5. Run `zonal_stats` to compute: `mean, min, max, std, count, median` per CDP.
  6. Plot and save a state map colored by CDP `mean` RPS.
  7. Save and combine results into `rps_cdps_all_states.csv`.

- **Outputs**:
  - Maps in `output_map_dir` named `rps_cdps_<State>.png`
  - Combined CSV in `output_csv_dir` named `rps_cdps_all_states.csv`

- **Adjustments**:
  - Change cores with `NUM_CORES` in the settings block.
  - Update paths at the top of the code block as needed.

- **Notes**:
  - CRS alignment ensures correct sampling against rasters.
  - Warnings for missing boundaries do not halt processing.

In [ ]:
################################################################################
#                            📦 IMPORT LIBRARIES                               #
################################################################################

import os
import glob
import re
import pandas as pd
import geopandas as gpd
import rasterio
import matplotlib.pyplot as plt
from rasterstats import zonal_stats
from multiprocessing import Pool
from datetime import datetime
import pytz

################################################################################
#                            🔧 SETTINGS & CONFIGURATION                       #
################################################################################

# =============================================================================
# ⏰ Timezone Configuration (Pacific Time for timestamps)
# =============================================================================
TIMEZONE = pytz.timezone('America/Los_Angeles')

def print_timestamped(message):
    """Print message with Pacific Time timestamp."""
    timestamp = datetime.now(TIMEZONE).strftime('%Y-%m-%d %H:%M:%S %Z')
    print(f"[{timestamp}] {message}")

# =============================================================================
# 🖥️ Parallel Processing Settings
# =============================================================================
NUM_CORES = 60  # Number of cores to use for parallel processing

################################################################################
#                            📂 FILE PATHS                                      #
################################################################################
# ┌─────────────────────────────────────────────────────────────────────────┐
# │  Configure paths to point to your data directories.                    │
# │  Override with environment variables before running:                    │
# │      export NETWORKS_DATA_DIR="/your/path/to/data"                      │
# │      export NETWORKS_RASTERS_DIR="/your/path/to/wildfire-risk-layers"  │
# │  Or edit the default paths below.                                       │
# └─────────────────────────────────────────────────────────────────────────┘
BASE_DATA_DIR    = os.environ.get("NETWORKS_DATA_DIR",    os.path.expanduser("~/data/networks_paper"))
RASTERS_DATA_DIR = os.environ.get("NETWORKS_RASTERS_DIR", os.path.expanduser("~/data/wildfire-risk-layers"))

# =============================================================================
# 📥 INPUT PATHS
# =============================================================================
raster_root    = os.path.join(RASTERS_DATA_DIR, "unzipped-states")
shapefile_root = os.path.join(BASE_DATA_DIR, "us-census-designated-places")

# CONUS RPS raster
rps_conus_raster_path = os.path.join(raster_root, "RDS-2020-0016-2__RPS_CONUS/RPS_CONUS/RPS_CONUS.tif")

# Dictionary mapping non-CONUS states to their RPS raster paths
state_raster_map = {
    "Alaska": os.path.join(raster_root, "RDS-2020-0016-2_Alaska/AK/RPS_AK.tif"),
    "District_of_Columbia": os.path.join(raster_root, "RDS-2020-0016-2_DistrictOfColumbia/DC/RPS_DC.tif"),
    "Hawaii": os.path.join(raster_root, "RDS-2020-0016-2_Hawaii/RDS-2020-0016-2_Hawaii/HI/RPS_HI.tif")
}

# =============================================================================
# 📤 OUTPUT PATHS
# =============================================================================
output_csv_dir = os.path.join(BASE_DATA_DIR, "output_csvs", "rps_cdp")
output_map_dir = os.path.join(BASE_DATA_DIR, "output_maps", "_state_wide_maps")

os.makedirs(output_csv_dir, exist_ok=True)
os.makedirs(output_map_dir, exist_ok=True)

# -------------------------------
# Load U.S. state boundaries from the Census TIGER/Line shapefile (500k resolution)
# -------------------------------
state_boundaries_url = "https://www2.census.gov/geo/tiger/GENZ2018/shp/cb_2018_us_state_500k.zip"
try:
    states_gdf = gpd.read_file(state_boundaries_url)
    print("Loaded state boundaries from Census TIGER/Line.")
except Exception as e:
    print("Error loading state boundaries:", e)
    states_gdf = None

# We'll share the state boundaries with worker processes via a global variable.
def init_worker(shared_states):
    global global_states_gdf
    global_states_gdf = shared_states

# -------------------------------
# Define the processing function for one state folder
# -------------------------------
def process_state(state_dir):
    try:
        # Expect folder names like "Alabama_01" or "District_of_Columbia_11"
        state_folder_name = os.path.basename(state_dir)
        m = re.match(r"^(.*)_[0-9]+$", state_folder_name)
        if not m:
            print(f"Skipping folder {state_dir}: name does not match expected pattern.")
            return None

        # Extract state name and create display and matching versions.
        state_name = m.group(1)                      # e.g., "Alabama" or "District_of_Columbia"
        state_display_name = state_name.replace("_", " ")
        
        # -------------------------------
        # Find the shapefile for Census designated places in this folder.
        # -------------------------------
        shapefile_pattern = os.path.join(state_dir, "tl_2023_*_place.shp")
        shapefile_list = glob.glob(shapefile_pattern)
        if not shapefile_list:
            print(f"No shapefile found in {state_dir}. Skipping.")
            return None
        shapefile_path = shapefile_list[0]
        
        # -------------------------------
        # Determine which raster file to use for this state.
        # -------------------------------
        if state_name in state_raster_map:
            raster_path = state_raster_map[state_name]
        else:
            # Use the CONUS RPS raster for all other states
            raster_path = rps_conus_raster_path
        
        if not os.path.exists(raster_path):
            print(f"Raster file {raster_path} not found for state {state_name}. Skipping.")
            return None
        
        print(f"\nProcessing state: {state_display_name}")
        print(f"  Shapefile: {shapefile_path}")
        print(f"  Raster: {raster_path}")
        
        # -------------------------------
        # Load the CDP shapefile and reproject if needed.
        # -------------------------------
        try:
            gdf = gpd.read_file(shapefile_path)
        except Exception as e:
            print(f"Error reading shapefile {shapefile_path}: {e}")
            return None

        try:
            with rasterio.open(raster_path) as src:
                raster_crs = src.crs
                nodata_value = src.nodata
        except Exception as e:
            print(f"Error opening raster {raster_path}: {e}")
            return None

        if gdf.crs != raster_crs:
            gdf = gdf.to_crs(raster_crs)
            print("  Reprojected CDP shapefile to match raster CRS.")
        
        # -------------------------------
        # Compute zonal statistics.
        # -------------------------------
        stats = zonal_stats(
            gdf,
            raster_path,
            stats="mean min max std count median",
            nodata=nodata_value,
            geojson_out=True
        )
        if not stats:
            print(f"  No zonal stats returned for state {state_display_name}. Skipping.")
            return None
        gdf_stats = gpd.GeoDataFrame.from_features(stats)
        
        # -------------------------------
        # Plot the results with state boundary overlay.
        # -------------------------------
        fig, ax = plt.subplots(1, 1, figsize=(10, 8))
        cmap = "viridis"
        gdf_stats.plot(column="mean", cmap=cmap, legend=True, ax=ax,
                       legend_kwds={'label': "Mean RPS", 'shrink': 0.6})
        
        # Overlay state boundary using the global_states_gdf loaded by the initializer.
        if global_states_gdf is not None:
            state_boundary = global_states_gdf[global_states_gdf["NAME"].str.lower() == state_display_name.lower()]
            if not state_boundary.empty:
                if state_boundary.crs != raster_crs:
                    state_boundary = state_boundary.to_crs(raster_crs)
                state_boundary.boundary.plot(ax=ax, edgecolor="black", linewidth=2)
            else:
                print(f"  Warning: No state boundary found for {state_display_name}.")
        
        # Set a professional title with a newline to prevent overlap.
        ax.set_title(f"Mean Risk to Potential Structures for Census designated places\nin {state_display_name}",
                     fontsize=16, pad=15)
        ax.set_axis_off()  # Clean look
        
        # Save the map to file.
        state_name_no_underscore = state_name.replace("_", "")
        map_output_fp = os.path.join(output_map_dir, f"rps_cdps_{state_name_no_underscore}.png")
        plt.savefig(map_output_fp, dpi=300, bbox_inches="tight")
        plt.close(fig)
        print(f"  Map saved to {map_output_fp}")
        
        # -------------------------------
        # Prepare zonal stats DataFrame for combining later.
        # -------------------------------
        cols_to_keep = ['GEOID', 'min', 'max', 'mean', 'std', 'count', 'median']
        missing_cols = [col for col in cols_to_keep if col not in gdf_stats.columns]
        if missing_cols:
            print(f"  Warning: missing columns {missing_cols} in zonal stats for {state_display_name}. Skipping state CSV export.")
            return None
        df_final = gdf_stats[cols_to_keep].copy()

        # Rename the zonal stats columns with prefix "rps_"
        df_final.rename(columns={
            'min': 'rps_min',
            'max': 'rps_max',
            'mean': 'rps_mean',
            'std': 'rps_std',
            'count': 'rps_count',
            'median': 'rps_median'
        }, inplace=True)

        df_final["state"] = state_display_name
        
        # Return results: the DataFrame, state name, and map file path.
        return (df_final, state_display_name, map_output_fp)
    
    except Exception as ex:
        print(f"Error processing {state_dir}: {ex}")
        return None

# -------------------------------
# Main processing block
# -------------------------------
if __name__ == '__main__':
    # Gather state directories from the shapefile root.
    state_dirs = [os.path.join(shapefile_root, d) for d in os.listdir(shapefile_root)
                  if os.path.isdir(os.path.join(shapefile_root, d))]
    
    # Create a multiprocessing Pool and initialize each worker with the state boundaries.
    with Pool(NUM_CORES, initializer=init_worker, initargs=(states_gdf,)) as pool:
        results = pool.map(process_state, state_dirs)
    
    # Filter out any results that returned None.
    valid_results = [res for res in results if res is not None]
    
    # Combine DataFrames from each state into one large CSV.
    if valid_results:
        df_list = [res[0] for res in valid_results]
        combined_df = pd.concat(df_list, ignore_index=True)
        combined_csv = os.path.join(output_csv_dir, "rps_cdps_all_states.csv")
        combined_df.to_csv(combined_csv, index=False)
        print(f"\nCombined CSV saved to {combined_csv}")
    else:
        print("No state data processed.")
